# 🧱 Part 02: Build a BPE Tokenizer From Scratch

> **Previous context**: In Part 01 we tried character-level and word-level tokenizers. Character-level almost never breaks, but it is too fine-grained. Word-level feels clean, but it falls apart when you meet an unseen word.
>
> **Goal for this part**: Implement a mini BPE tokenizer with your own hands, and understand why it can grow from characters into subwords step by step.

## Learning map

Do not rush into code yet. The point of BPE is not memorizing a formula. It is holding on to 4 questions:

1. **Why start from characters?** Because characters are the most stable starting point and rarely cause OOV.
2. **Why merge the most frequent pair?** Because the most frequent adjacent pattern is the best compression target.
3. **Why do merge rules have an order?** Because when encoding new text, we must replay merges in the same order they were learned.
4. **Why is the vocab size controllable?** Because each merge adds exactly one new token.

One-sentence memory for BPE:

**Repeat: find the most frequent adjacent token pair, merge it into a new token.**

Think of it like building with bricks. At the start you only have small bricks (characters). Each step glues together the two bricks that most often appear next to each other, and slowly you grow subwords like `th`, `the `, `ing`.


## 1. Why BPE?

Let's make the trade-off explicit: **what is a tokenizer balancing?**

| Scheme | Pros | Cons |
|:---|:---|:---|
| Character-level | almost never hits unknown words | sequences are too long |
| Word-level | tokens look like human words | unseen words cause OOV |
| BPE (subword) | common words can merge, rare words can split | you must train merge rules |

So the BPE goal is very plain:

**Make common things bigger, keep uncommon things smaller.**

For example, `the` is common, so it is worth becoming a single token. But an unseen new word can still fall back to smaller pieces, instead of crashing.

We will start from a tiny corpus. A small corpus is better for learning: you can clearly see what happens at every merge step.


**Corpus: keep it tiny so you can watch every step**


In [1]:
corpus = [
    "the cat sat on the mat",
    "the dog sat on the log",
    "the cat and the dog",
    "i love my cat",
    "i love my dog",
    "the cat is cute",
    "the dog is happy",
    "the mat is soft",
    "the log is hard",
    "cats and dogs are friends",
]

print("Our corpus:")
for i, s in enumerate(corpus):
    print(f"  [{i}] {s}")


Our corpus:
  [0] the cat sat on the mat
  [1] the dog sat on the log
  [2] the cat and the dog
  [3] i love my cat
  [4] i love my dog
  [5] the cat is cute
  [6] the dog is happy
  [7] the mat is soft
  [8] the log is hard
  [9] cats and dogs are friends


## 2. Train BPE

### Starting point: characters

BPE starts from a **character-level** representation: split text into characters, treat each character as a token.

Why start from characters? Because characters are small but stable. As long as a character is in the vocab, a new word can still be represented by splitting it.

One detail: what about spaces?

Real GPT tokenizers treat spaces more carefully. To keep the main pipeline easy to see, our mini version treats a space `' '` as just another character.
That way you will clearly see that BPE does not only merge letters, it can also merge patterns like `e + space`.


In [2]:
# Step 1: split every sentence into a list of characters
# Note: a space ' ' is treated as a normal character.

def text_to_tokens(text):
    """Split a string into a character-level token list."""
    return list(text)

# Look at the first sentence
sentence = corpus[0]
chars = text_to_tokens(sentence)

print(f"Original text: '{sentence}'")
print(f"Character tokens: {chars}")
print(f"Total tokens: {len(chars)}")

# Convert every sentence in the corpus into a character list
corpus_tokens = [text_to_tokens(s) for s in corpus]
print(f"\nCharacter counts per sentence: {[len(t) for t in corpus_tokens]}")


Original text: 'the cat sat on the mat'
Character tokens: ['t', 'h', 'e', ' ', 'c', 'a', 't', ' ', 's', 'a', 't', ' ', 'o', 'n', ' ', 't', 'h', 'e', ' ', 'm', 'a', 't']
Total tokens: 22

Character counts per sentence: [22, 22, 19, 13, 13, 15, 16, 15, 15, 25]


### Count adjacent pairs

Now we ask a very mechanical question: **which two adjacent tokens appear together most often?**

For a single sequence:

```
['t', 'h', 'e', ' ', 'c', 'a', 't']
  ^--^
 ('t','h'): 1 time
      ^--^
     ('h','e'): 1 time
          ^--^
         ('e',' '): 1 time
              ^--^
             (' ','c'): 1 time
                  ^--^
                 ('c','a'): 1 time
                      ^--^
                     ('a','t'): 1 time
```

A single sentence is not enough to show patterns, so we count pairs across the **entire corpus**.
Whatever pair appears the most is the best candidate to merge first.


In [3]:
from collections import defaultdict

def count_pairs(token_lists):
    """Count frequencies of adjacent token pairs across the whole corpus.

    Args:
        token_lists: List[List[str]]. Each sentence is represented as a token list.

    Returns:
        dict mapping (token_a, token_b) -> count.
    """
    pairs = defaultdict(int)
    for tokens in token_lists:
        for i in range(len(tokens) - 1):
            pair = (tokens[i], tokens[i + 1])
            pairs[pair] += 1
    return dict(pairs)

# Count pair frequencies in the initial state (pure character-level)
initial_pairs = count_pairs(corpus_tokens)

print("=== Adjacent pair frequencies (initial character-level state) ===")
# Print from high to low frequency
for pair, count in sorted(initial_pairs.items(), key=lambda x: -x[1]):
    print(f"  {pair}: {count} times")

print(f"\nMost frequent pair: {max(initial_pairs, key=initial_pairs.get)}")
print(f"Appears {initial_pairs[max(initial_pairs, key=initial_pairs.get)]} times")


=== Adjacent pair frequencies (initial character-level state) ===
  ('e', ' '): 13 times
  ('t', 'h'): 10 times
  ('h', 'e'): 10 times
  ('a', 't'): 9 times
  ('o', 'g'): 7 times
  ('t', ' '): 6 times
  ('s', ' '): 6 times
  (' ', 'c'): 5 times
  ('c', 'a'): 5 times
  (' ', 'd'): 5 times
  ('d', 'o'): 5 times
  (' ', 'm'): 4 times
  (' ', 'l'): 4 times
  ('l', 'o'): 4 times
  (' ', 'i'): 4 times
  ('i', 's'): 4 times
  (' ', 's'): 3 times
  (' ', 't'): 3 times
  ('g', ' '): 3 times
  (' ', 'a'): 3 times
  ('n', 'd'): 3 times
  ('s', 'a'): 2 times
  (' ', 'o'): 2 times
  ('o', 'n'): 2 times
  ('n', ' '): 2 times
  ('m', 'a'): 2 times
  ('a', 'n'): 2 times
  ('d', ' '): 2 times
  ('i', ' '): 2 times
  ('o', 'v'): 2 times
  ('v', 'e'): 2 times
  ('m', 'y'): 2 times
  ('y', ' '): 2 times
  (' ', 'h'): 2 times
  ('h', 'a'): 2 times
  ('a', 'r'): 2 times
  ('c', 'u'): 1 times
  ('u', 't'): 1 times
  ('t', 'e'): 1 times
  ('a', 'p'): 1 times
  ('p', 'p'): 1 times
  ('p', 'y'): 1 times
  ('s',

### Merge the most frequent pair

Once we find the top pair, BPE merges **every occurrence of that pair** in the corpus.

```
Before: ['t', 'h', 'e', ' ', 'c', 'a', 't']
Merge:  ('t', 'h')
After:  ['th', 'e', ' ', 'c', 'a', 't']
          ^  't' and 'h' become one new token 'th'
```

This is like welding two small parts into a bigger part. Next time we count pairs, `th` participates as a single unit.


In [4]:
def merge_pair(token_lists, pair_to_merge):
    """Merge every occurrence of a given adjacent pair in the corpus.

    Args:
        token_lists: List[List[str]]. Current tokenized corpus.
        pair_to_merge: (str, str). The pair to merge.

    Returns:
        (new_token_lists, new_token)
    """
    a, b = pair_to_merge
    new_token = a + b  # the new token is just string concatenation

    new_token_lists = []
    for tokens in token_lists:
        new_tokens = []
        i = 0
        while i < len(tokens):
            # If current position + next position matches the pair, merge!
            if i < len(tokens) - 1 and tokens[i] == a and tokens[i + 1] == b:
                new_tokens.append(new_token)
                i += 2  # consume two tokens
            else:
                new_tokens.append(tokens[i])
                i += 1
        new_token_lists.append(new_tokens)

    return new_token_lists, new_token

# Demo: do one merge
best_pair = max(initial_pairs, key=initial_pairs.get)
print(f"Pair to merge: {best_pair}")

merged_tokens, new_token = merge_pair(corpus_tokens, best_pair)
print(f"New token: '{new_token}'")
print(f"\nToken sequences after the merge:")
for i, tokens in enumerate(merged_tokens):
    print(f"  [{i}] {tokens}")


Pair to merge: ('e', ' ')
New token: 'e '

Token sequences after the merge:
  [0] ['t', 'h', 'e ', 'c', 'a', 't', ' ', 's', 'a', 't', ' ', 'o', 'n', ' ', 't', 'h', 'e ', 'm', 'a', 't']
  [1] ['t', 'h', 'e ', 'd', 'o', 'g', ' ', 's', 'a', 't', ' ', 'o', 'n', ' ', 't', 'h', 'e ', 'l', 'o', 'g']
  [2] ['t', 'h', 'e ', 'c', 'a', 't', ' ', 'a', 'n', 'd', ' ', 't', 'h', 'e ', 'd', 'o', 'g']
  [3] ['i', ' ', 'l', 'o', 'v', 'e ', 'm', 'y', ' ', 'c', 'a', 't']
  [4] ['i', ' ', 'l', 'o', 'v', 'e ', 'm', 'y', ' ', 'd', 'o', 'g']
  [5] ['t', 'h', 'e ', 'c', 'a', 't', ' ', 'i', 's', ' ', 'c', 'u', 't', 'e']
  [6] ['t', 'h', 'e ', 'd', 'o', 'g', ' ', 'i', 's', ' ', 'h', 'a', 'p', 'p', 'y']
  [7] ['t', 'h', 'e ', 'm', 'a', 't', ' ', 'i', 's', ' ', 's', 'o', 'f', 't']
  [8] ['t', 'h', 'e ', 'l', 'o', 'g', ' ', 'i', 's', ' ', 'h', 'a', 'r', 'd']
  [9] ['c', 'a', 't', 's', ' ', 'a', 'n', 'd', ' ', 'd', 'o', 'g', 's', ' ', 'a', 'r', 'e ', 'f', 'r', 'i', 'e', 'n', 'd', 's']


**After one merge, the world changes**

Notice: a merge does not only change the vocab. It also changes what we count next.

The whole BPE training loop is just 4 steps:

1. Count pair frequencies
2. Find the most frequent pair
3. Merge that pair
4. Record the merge as a **merge rule**

Then repeat from step 1.

Every merge creates new adjacent relationships. If we first get `th`, we might later get `the `.
BPE is not "recognizing English words" in one shot. It grows them step by step.


## 3. Training log

Now we will run the loop for real.

There is one key parameter: `num_merges`.

- `num_merges=10`: add 10 new tokens; small vocab; text stays more fragmented
- `num_merges=50`: add 50 new tokens; larger vocab; common chunks become more intact
- Real LLM tokenizers: often do many, many merges; vocab size can be tens of thousands

We will start with `num_merges=15`. Do not stare at the final vocab yet. The point is the log: what happens at each step.


In [5]:
class BPETokenizer:
    """
    Complete mini BPE tokenizer implementation

    Three core functions:
    1. train()  - learn merge rules from a corpus
    2. encode() - text -> list of token IDs
    3. decode() - list of token IDs -> text
    """

    def __init__(self):
        # merge_rules: record each merged pair in order
        # This is the core BPE data structure. During encode(), we greedily apply rules in this order.
        self.merge_rules = []
        # Final vocab: base characters + new tokens learned by merges
        self.vocab = set()
        self.stoi = {}
        self.itos = {}

    def train(self, texts, num_merges=15, verbose=True):
        """
        BPE training

        Args:
            texts: corpus list
            num_merges: number of merges
            verbose: whether to print each step
        """
        # Step 0: initialize as character-level
        token_lists = [list(text) for text in texts]
        base_vocab = set(c for tokens in token_lists for c in tokens)
        learned_vocab = set(base_vocab)

        if verbose:
            print(f"{'='*60}")
            print(f"BPE training starts! Initial state: split each sentence into characters")
            print(f"{'='*60}")
            print(f"Initial charset size: {len(set(c for t in token_lists for c in t))}")
            print()

        for step in range(num_merges):
            # Step 1: count pair frequencies
            pairs = defaultdict(int)
            for tokens in token_lists:
                for i in range(len(tokens) - 1):
                    pairs[(tokens[i], tokens[i + 1])] += 1

            if not pairs:
                break

            # Step 2: find the most frequent pair
            best_pair = max(pairs, key=pairs.get)
            best_count = pairs[best_pair]

            # Step 3: record merge rule
            self.merge_rules.append(best_pair)

            # Step 4: merge
            a, b = best_pair
            new_token = a + b

            new_token_lists = []
            for tokens in token_lists:
                new_tokens = []
                i = 0
                while i < len(tokens):
                    if i < len(tokens) - 1 and tokens[i] == a and tokens[i + 1] == b:
                        new_tokens.append(new_token)
                        i += 2
                    else:
                        new_tokens.append(tokens[i])
                        i += 1
                new_token_lists.append(new_tokens)

            token_lists = new_token_lists

            learned_vocab.add(new_token)

            if verbose:
                print(f"Step {step+1:2d}: merge {best_pair} -> '{new_token}'  (appears {best_count} times) | vocab size now: {len(learned_vocab)}")

        # Build final vocab
        # Note: we cannot keep only tokens that still appear in the final corpus.
        # When encoding unseen combinations, we must be able to fall back to base characters.
        self.vocab = learned_vocab
        sorted_vocab = sorted(list(self.vocab))
        self.stoi = {t: i for i, t in enumerate(sorted_vocab)}
        self.itos = {i: t for i, t in enumerate(sorted_vocab)}

        if verbose:
            print(f"\n{'='*60}")
            print(f"Training finished!")
            print(f"  - merges performed: {len(self.merge_rules)}")
            print(f"  - final vocab size: {len(self.vocab)} tokens")
            print(f"  - merge rules: {self.merge_rules}")
            print(f"{'='*60}")

        return token_lists


In [6]:
# Train the BPE tokenizer: 15 merge steps, print every step
bpe = BPETokenizer()
final_tokens = bpe.train(corpus, num_merges=15, verbose=True)


BPE training starts! Initial state: split each sentence into characters
Initial charset size: 20

Step  1: merge ('e', ' ') -> 'e '  (appears 13 times) | vocab size now: 21
Step  2: merge ('t', 'h') -> 'th'  (appears 10 times) | vocab size now: 22
Step  3: merge ('th', 'e ') -> 'the '  (appears 10 times) | vocab size now: 23
Step  4: merge ('a', 't') -> 'at'  (appears 9 times) | vocab size now: 24
Step  5: merge ('o', 'g') -> 'og'  (appears 7 times) | vocab size now: 25
Step  6: merge ('at', ' ') -> 'at '  (appears 6 times) | vocab size now: 26
Step  7: merge ('s', ' ') -> 's '  (appears 6 times) | vocab size now: 27
Step  8: merge ('d', 'og') -> 'dog'  (appears 5 times) | vocab size now: 28
Step  9: merge ('i', 's ') -> 'is '  (appears 4 times) | vocab size now: 29
Step 10: merge ('the ', 'c') -> 'the c'  (appears 3 times) | vocab size now: 30
Step 11: merge ('the c', 'at ') -> 'the cat '  (appears 3 times) | vocab size now: 31
Step 12: merge (' ', 'the ') -> ' the '  (appears 3 times

**Keypoint 4: when reading the log, only watch what "grows" first**

Looking back at the training log, it feels like a growth process:

```
Step 1: ('e', ' ')   -> 'e '     <- "e + space" is very common
Step 2: ('t', 'h')   -> 'th'     <- "the" is common, so "th" grows early
Step 3: ('th', 'e ') -> 'the '   <- "th" + "e " becomes "the "
Step 4: ('a', 't')   -> 'at'     <- cat / sat / mat makes "at" frequent
Step 5: ('o', 'g')   -> 'og'     <- dog / log makes "og" frequent too
```

**Key observation**: BPE is not a program that "understands English".
It is brutally honest statistics: if two things often sit together, they get merged earlier.

This also explains why vocab size is controllable: if you say "do 15 merges", it adds exactly 15 new tokens.


## 4. Encode

### Merge in order

After training, we have an ordered list of merge rules. Now we receive new text. How do we encode it?

The pipeline is simple:

1. Split new text into characters
2. Replay merge rules **in the exact training order**, one by one
3. Finally map the remaining tokens to IDs

```
"the cat"
  -> ['t','h','e',' ','c','a','t']
  -> apply rule 1: ('e',' ') -> 'e '
  -> ['t','h','e ','c','a','t']
  -> apply rule 2: ('t','h') -> 'th'
  -> ['th','e ','c','a','t']
  -> apply rule 3: ('th','e ') -> 'the '
  -> ['the ','c','a','t']
  -> apply rule 4: ('a','t') -> 'at'
  -> ['the ','c','at']
```

The easiest pitfall is: **you cannot shuffle the order.**

Because what you merge first changes what can be merged later.
Encoding is not re-counting statistics. It is replaying the learned rules.


In [7]:
# Add an encode method to BPETokenizer
def bpe_encode(self, text):
    """BPE encode: text -> list of token IDs.

    Core idea: replay merge rules in the training order, greedily merging.
    """
    # Step 1: split into characters
    tokens = list(text)

    # Step 2: apply each merge rule in order
    for (a, b) in self.merge_rules:
        new_tokens = []
        i = 0
        while i < len(tokens):
            if i < len(tokens) - 1 and tokens[i] == a and tokens[i + 1] == b:
                new_tokens.append(a + b)
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1
        tokens = new_tokens

    # Step 3: map each token to an ID
    ids = []
    for token in tokens:
        if token in self.stoi:
            ids.append(self.stoi[token])
        else:
            # Unseen token -> fall back to characters (this is why BPE usually does not crash)
            for ch in token:
                ids.append(self.stoi[ch])
    return ids

# Monkey-patch the method
BPETokenizer.encode = bpe_encode

print("Added encode()!")


Added encode()!


In [8]:
# Test encode
text = "the cat"
ids = bpe.encode(text)
print(f"Original text: '{text}'")
print(f"Token IDs: {ids}")
print(f"Number of tokens: {len(ids)}")

# Explain each ID
print(f"\nExplain each token:")
for i, tid in enumerate(ids):
    print(f"  ID {tid} -> '{bpe.itos[tid]}'")

# Compare against character-level
print(f"\nCompare: text characters = {len(text)}, BPE tokens = {len(ids)}, compression ratio = {len(text)/len(ids):.1f}x")


Original text: 'the cat'
Token IDs: [30, 3]
Number of tokens: 2

Explain each token:
  ID 30 -> 'the c'
  ID 3 -> 'at'

Compare: text characters = 7, BPE tokens = 2, compression ratio = 3.5x


**Keypoint 6: decode is just lookup + concatenate**

Decode is much simpler than encode.

It does not count and does not merge. It does one thing: look up each token ID back to a string, then concatenate.

You can think of encode as "split and assign IDs", and decode as "recover pieces by IDs and glue them back".


In [9]:
def bpe_decode(self, ids):
    """BPE decode: list of token IDs -> text.

    This is just lookup + concatenation. No counting, no merging.
    """
    return "".join([self.itos[i] for i in ids])

BPETokenizer.decode = bpe_decode

# Test full encode/decode
for test_text in ["the cat", "my dog is happy", "i love cats"]:
    ids = bpe.encode(test_text)
    recovered = bpe.decode(ids)
    status = "OK" if test_text == recovered else "Mismatch"
    print(f"{status} | '{test_text}' -> {ids} -> '{recovered}'")


OK | 'the cat' -> [30, 3] -> 'the cat'
OK | 'my dog is happy' -> [16, 34, 0, 7, 0, 14, 12, 2, 21, 21, 34] -> 'my dog is happy'
OK | 'i love cats' -> [13, 0, 15, 19, 33, 9, 5, 3, 23] -> 'i love cats'


**Keypoint 7: why BPE does not fear most unseen words**

Look at this part of the encoding logic:

```python
if token in self.stoi:
    ids.append(self.stoi[token])
else:
    for ch in token:          # key: fall back to characters
        ids.append(self.stoi[ch])
```

This is why BPE is more robust than a word-level tokenizer.

If a whole word was never seen, it does not give up. It falls back to smaller subwords, or even characters.

One caveat: our mini version is **character-level BPE**. If the new text contains a character that never appears in the training corpus (for example the corpus never contained `z`), it can still error.

Industrial tokenizers usually start from bytes. With only 256 byte values, they can cover any Unicode text, so they are even more robust.


In [10]:
# Demo: encode a sentence not present in the training corpus
unseen = "a cat runs fast"
print(f"Test unseen sentence: '{unseen}'")
print("'runs fast' never appeared in the training corpus, but all characters did.")

ids = bpe.encode(unseen)
recovered = bpe.decode(ids)

print(f"\nToken IDs: {ids}")
print(f"Decoded back: '{recovered}'")
print(f"Status: {'OK (no crash)' if unseen == recovered else 'Warning'}")

# Show what tokens it became
print(f"\nTokens:")
for tid in ids:
    print(f"  '{bpe.itos[tid]}'", end="")
print()


Test unseen sentence: 'a cat runs fast'
'runs fast' never appeared in the training corpus, but all characters did.

Token IDs: [2, 0, 5, 4, 22, 32, 17, 24, 10, 2, 23, 27]
Decoded back: 'a cat runs fast'
Status: OK (no crash)

Tokens:
  'a'  ' '  'c'  'at '  'r'  'u'  'n'  's '  'f'  'a'  's'  't'


## 5. Number of merges

More merges -> larger vocab -> tokens become more "intact".

But bigger is not always better. If the vocab is too small, sequences get long. If the vocab is too large, many tokens are rare and not worth learning.

Let's run a small experiment: for the same sentence, how does tokenization change under different merge counts?


In [11]:
# Train multiple tokenizers with different merge counts
for n_merges in [5, 15, 30]:
    t = BPETokenizer()
    t.train(corpus, num_merges=n_merges, verbose=False)

    test = "the cat sat on the mat"
    ids = t.encode(test)

    print(f"merges={n_merges:2d} | vocab={len(t.vocab):2d} | tokens={len(ids):2d} | tokens: {[t.itos[i] for i in ids]}")

print()
print("Observation: more merges -> common chunks are more likely to stay intact -> fewer tokens.")


merges= 5 | vocab=25 | tokens=13 | tokens: ['the ', 'c', 'at', ' ', 's', 'at', ' ', 'o', 'n', ' ', 'the ', 'm', 'at']
merges=15 | vocab=35 | tokens= 6 | tokens: ['the cat ', 'sat o', 'n', ' the ', 'm', 'at']
merges=30 | vocab=50 | tokens= 4 | tokens: ['the cat ', 'sat on the ', 'm', 'at']

Observation: more merges -> common chunks are more likely to stay intact -> fewer tokens.


## 6. Real-world BPE

### How tokenization affects what the model sees

BPE is practical, but it is not a perfect answer. It compresses text into tokens, and compression can change the structure the model sees.

**Example 1: numbers may get hidden**

A number like `327` might become a single token, not the characters `3`, `2`, `7`. Then the model does not automatically see hundreds/tens/ones structure.
So when it does `327 + 1`, it may not naturally "carry" from the ones place the way you do.

**Example 2: spaces are sensitive in code**

Python indentation matters. If spaces and newlines are tokenized in unstable ways, learning code structure becomes harder.
That is why code-focused models often optimize tokens related to spaces, newlines, and indentation.

**Example 3: why not just use characters?**

Character- or byte-level tokenization is more lossless: digits and whitespace structure are clearer.
But sequences become much longer. Transformers are expensive on long sequences because Self-Attention is usually quadratic in sequence length.

So the mainstream choice is still: BPE (or similar methods) as a compromise between information fidelity and compute cost.


**Keypoint 9: industrial BPE is more robust and faster than our mini version**

Our implementation is a teaching version. Real tokenizers do many engineering optimizations:

| Our version | Industrial version |
|:---|:---|
| start from characters | start from **bytes**, covering all Unicode |
| treat space as a normal character | special handling for leading spaces, newlines, indentation |
| count pairs naively | faster data structures and algorithms |
| tiny corpus | massive corpus |
| pure Python | often C++ / Rust |

What does "start from bytes" mean?

Character-level looks stable, but Unicode is huge. For example:

```text
The character "€" uses 3 UTF-8 bytes: E4 BD A0
The emoji "😊" uses 4 UTF-8 bytes: F0 9F 98 8A
```

If a tokenizer starts from characters, it must already know every character it may meet (Chinese, emoji, symbols...).
If it starts from bytes, the starting vocab always has only 256 items: `0` to `255`.
So no matter what Unicode you input, it can at least be represented as bytes without immediately crashing due to OOV.

In other words:

```text
Character-level: first ask "do I know this character?"
Byte-level: any character becomes bytes first, so there is always an entry point
```

But the core idea is unchanged: **count pairs -> merge the most frequent -> repeat -> obtain ordered merge rules**.

After this part, if you open GPT-2's `encoder.json` and `vocab.bpe`, they should feel much less like a black box.


### Load a real tokenizer

Below we use `tiktoken` to load GPT-2's real byte-level BPE tokenizer.

You do not need to understand all engineering details yet. Just look at three facts:

1. It really does split text into tokens
2. Every token has a fixed ID
3. How spaces, numbers, newlines, Chinese, and emoji are split is different from our mini version


In [12]:
# Load the real GPT-2 tokenizer (byte-level BPE)
try:
    import tiktoken

    real_tokenizer_name = "gpt2"
    real_tokenizer = tiktoken.get_encoding(real_tokenizer_name)
    print(f"Real tokenizer: {real_tokenizer_name}")
    print(f"Vocab size: {real_tokenizer.n_vocab}")
except Exception as e:
    real_tokenizer = None
    print("Could not load GPT-2 tokenizer via tiktoken")
    print(f"Reason: {e}")
    print("After you install and cache tiktoken, this cell will print real-tokenizer results.")


def show_real_tokenization(text):
    """Print how a real tokenizer splits text into tokens and IDs."""
    ids = real_tokenizer.encode(text)
    tokens = [real_tokenizer.decode([tok_id]) for tok_id in ids]

    print(f"Text: {text!r}")
    print(f"Number of tokens: {len(ids)}")
    for i, (tok_id, token) in enumerate(zip(ids, tokens)):
        visible = token.replace("\n", "\\n").replace("\t", "\\t")
        token_bytes = list(real_tokenizer.decode_single_token_bytes(tok_id))
        print(f"  {i:02d} | id={tok_id:<6} | token={visible!r} | bytes={token_bytes}")
    return ids, tokens


if real_tokenizer is not None:
    show_real_tokenization("the cat sat on the mat")


Real tokenizer: gpt2
Vocab size: 50257
Text: 'the cat sat on the mat'
Number of tokens: 6
  00 | id=1169   | token='the' | bytes=[116, 104, 101]
  01 | id=3797   | token=' cat' | bytes=[32, 99, 97, 116]
  02 | id=3332   | token=' sat' | bytes=[32, 115, 97, 116]
  03 | id=319    | token=' on' | bytes=[32, 111, 110]
  04 | id=262    | token=' the' | bytes=[32, 116, 104, 101]
  05 | id=2603   | token=' mat' | bytes=[32, 109, 97, 116]


### Compare against our hand-written BPE

Our mini BPE was trained on only 10 English sentences, so its vocab is tiny.
The real tokenizer is trained on massive corpora, and starts from bytes, so it can handle arbitrary Unicode text.


In [13]:
def try_mini_bpe(text):
    """Encode with our mini BPE; on failure return the exception."""
    try:
        ids = bpe.encode(text)
        tokens = [bpe.itos[i] for i in ids]
        return ids, tokens, None
    except Exception as e:
        return None, None, e


compare_texts = [
    "the cat sat on the mat",
    " the cat",
    "the  cat",
    "327 + 1 = 328",
    "def f():\n    return x",
    "Hello, € world 🙂",
]

if real_tokenizer is not None:
    for text in compare_texts:
        print("=" * 68)
        print(f"Text: {text!r}")

        mini_ids, mini_tokens, error = try_mini_bpe(text)
        if error is None:
            print(f"mini BPE: {len(mini_ids)} tokens")
            print(f"  tokens: {mini_tokens}")
            print(f"  ids:    {mini_ids}")
        else:
            print("mini BPE: cannot encode")
            print(f"  reason: mini vocab is too small, error={error!r}")

        real_ids = real_tokenizer.encode(text)
        real_tokens = [real_tokenizer.decode([tok_id]) for tok_id in real_ids]
        real_tokens = [t.replace("\n", "\\n") for t in real_tokens]
        print(f"real GPT-2 BPE: {len(real_ids)} tokens")
        print(f"  tokens: {real_tokens}")
        print(f"  ids:    {real_ids}")

    print("=" * 68)
    print("Key observation: the real tokenizer is not only larger; it also handles bytes, spaces, newlines, and Unicode.")
    print("Our mini BPE is for understanding merge rules; industrial tokenizers solve coverage, speed, and edge cases.")


Text: 'the cat sat on the mat'
mini BPE: 6 tokens
  tokens: ['the cat ', 'sat o', 'n', ' the ', 'm', 'at']
  ids:    [31, 26, 17, 1, 16, 3]
real GPT-2 BPE: 6 tokens
  tokens: ['the', ' cat', ' sat', ' on', ' the', ' mat']
  ids:    [1169, 3797, 3332, 319, 262, 2603]
Text: ' the cat'
mini BPE: 3 tokens
  tokens: [' ', 'the c', 'at']
  ids:    [0, 30, 3]
real GPT-2 BPE: 2 tokens
  tokens: [' the', ' cat']
  ids:    [262, 3797]
Text: 'the  cat'
mini BPE: 4 tokens
  tokens: ['the ', ' ', 'c', 'at']
  ids:    [29, 0, 5, 3]
real GPT-2 BPE: 3 tokens
  tokens: ['the', ' ', ' cat']
  ids:    [1169, 220, 3797]
Text: '327 + 1 = 328'
mini BPE: cannot encode
  reason: mini vocab is too small, error=KeyError('3')
real GPT-2 BPE: 5 tokens
  tokens: ['327', ' +', ' 1', ' =', ' 328']
  ids:    [34159, 1343, 352, 796, 39093]
Text: 'def f():\n    return x'
mini BPE: cannot encode
  reason: mini vocab is too small, error=KeyError('(')
real GPT-2 BPE: 9 tokens
  tokens: ['def', ' f', '():', '\\n', ' ', ' '

### Special cases: numbers, spaces, indentation, and special tokens

Many "weird" tokenization behaviors come from training data and engineering choices.
The outputs below matter: they explain why tokenization affects math, code, and mixed Chinese/English input.


In [14]:
special_cases = [
    ("numbers", "327 + 1 = 328"),
    ("leading-space behavior", "cat"),
    ("same word with a leading space", " cat"),
    ("multiple spaces", "the    cat"),
    ("newlines and indentation", "def f():\n    return x"),
    ("Chinese and emoji", "Hello, € world 🙂"),
]

if real_tokenizer is not None:
    for label, text in special_cases:
        print("=" * 68)
        print(f"Special case: {label}")
        show_real_tokenization(text)

    print("=" * 68)
    print("Special token: <|endoftext|>")
    text = "hello<|endoftext|>world"

    try:
        real_tokenizer.encode(text)
    except ValueError as e:
        print("By default, encode() refuses to treat a special token as normal text.")
        print(f"Error summary: {str(e).splitlines()[0]}")

    ids = real_tokenizer.encode(text, allowed_special={"<|endoftext|>"})
    tokens = [real_tokenizer.decode([tok_id]) for tok_id in ids]
    print("After allowing the special token:")
    print(f"  ids:    {ids}")
    print(f"  tokens: {tokens}")
    print("Key point: special tokens are extra control symbols; they are not learned as ordinary BPE merge rules.")


Special case: numbers
Text: '327 + 1 = 328'
Number of tokens: 5
  00 | id=34159  | token='327' | bytes=[51, 50, 55]
  01 | id=1343   | token=' +' | bytes=[32, 43]
  02 | id=352    | token=' 1' | bytes=[32, 49]
  03 | id=796    | token=' =' | bytes=[32, 61]
  04 | id=39093  | token=' 328' | bytes=[32, 51, 50, 56]
Special case: leading-space behavior
Text: 'cat'
Number of tokens: 1
  00 | id=9246   | token='cat' | bytes=[99, 97, 116]
Special case: same word with a leading space
Text: ' cat'
Number of tokens: 1
  00 | id=3797   | token=' cat' | bytes=[32, 99, 97, 116]
Special case: multiple spaces
Text: 'the    cat'
Number of tokens: 5
  00 | id=1169   | token='the' | bytes=[116, 104, 101]
  01 | id=220    | token=' ' | bytes=[32]
  02 | id=220    | token=' ' | bytes=[32]
  03 | id=220    | token=' ' | bytes=[32]
  04 | id=3797   | token=' cat' | bytes=[32, 99, 97, 116]
Special case: newlines and indentation
Text: 'def f():\n    return x'
Number of tokens: 9
  00 | id=4299   | token='def'

### A fun question: why does a model sometimes think `1.11` is larger than `1.9`?

Before reading the explanation, guess first:

```text
Between 1.11 and 1.9, which is larger?
```

Mathematically, `1.9` is larger because you can pad zeros:

```text
1.9  = 1.90
1.11 = 1.11
So 1.90 > 1.11
```

But an LLM does not start by treating `1.11` as a "decimal number object".
What it first sees is the token ID sequence.

If tokenization looks like:

```text
"1.11" -> ["1", ".", "11"]
"1.9"  -> ["1", ".", "9"]
```

The model can be misled by surface patterns: `11` looks larger than `9`, so it may guess `1.11 > 1.9`.

This does not mean the model can never compare decimals. It means it must learn the rule of **aligning decimal places** from token sequences.
If training data lacks such cases, or the prompt looks like string comparison, the model can slip.


In [15]:
# Fun observation: how does the real tokenizer split 1.11 vs 1.9?
number_texts = ["1.11", "1.9"]

if real_tokenizer is not None:
    for text in number_texts:
        ids = real_tokenizer.encode(text)
        tokens = [real_tokenizer.decode([tok_id]) for tok_id in ids]
        print(f"{text!r} -> tokens={tokens}, ids={ids}")

    print()
    print("Math comparison:")
    print(f"  1.11 > 1.9  ? {1.11 > 1.9}")
    print(f"  1.90 > 1.11 ? {1.90 > 1.11}")
    print()
    print("Key observation: the model does not see floats; it sees token sequences.")
    print("To compare decimals, it must learn digit alignment, not just compare surface tokens like '11' vs '9'.")


'1.11' -> tokens=['1', '.', '11'], ids=[16, 13, 1157]
'1.9' -> tokens=['1', '.', '9'], ids=[16, 13, 24]

Math comparison:
  1.11 > 1.9  ? False
  1.90 > 1.11 ? True

Key observation: the model does not see floats; it sees token sequences.
To compare decimals, it must learn digit alignment, not just compare surface tokens like '11' vs '9'.


**Keypoint 10: special tokens are not learned by BPE merge rules**

Just remember the boundary for now:

```text
BPE-learned tokens:           th + e -> the
Hand-defined control symbols: <BOS>, <EOS>, <PAD>, <think>, </think>
```

BPE splits normal text into tokens. Special tokens tell the model "where it starts", "where it ends", "what is padding", and "what is the thinking region".

So they are usually not learned by counting pairs. They are appended to the vocab when training or extending a tokenizer.
We will discuss how they are used in Mini-GPT and in the thinking-model part.


**Visualization: how merge rules grow step by step**

One last small bonus: we will visualize the merge rules.

This is not for show. It helps lock in the intuition: common short patterns appear first, and longer patterns grow on top of them.


In [16]:
print("Full merge rule list (in training order):")
print("Watch how 'the' is formed:")
print()

for i, (a, b) in enumerate(bpe.merge_rules):
    arrow = ""
    # Mark merges related to 'the'
    merged = a + b
    if merged in ['th', 'the ', 'the c', 'the cat ']:
        arrow = "  <- how 'the' grows"
    if a in ['th', 'the ', 'the c'] or b in ['e ', 'c', 'at ']:
        arrow = "  <- how 'the' grows"
    print(f"  Rule {i+1:2d}: '{a}' + '{b}' -> '{merged}'{arrow}")

print()
print("See? 'the ' is not recognized in one shot. It grows as 't'+'h'->'th', then merges with 'e ', step by step.")


Full merge rule list (in training order):
Watch how 'the' is formed:

  Rule  1: 'e' + ' ' -> 'e '
  Rule  2: 't' + 'h' -> 'th'  <- how 'the' grows
  Rule  3: 'th' + 'e ' -> 'the '  <- how 'the' grows
  Rule  4: 'a' + 't' -> 'at'
  Rule  5: 'o' + 'g' -> 'og'
  Rule  6: 'at' + ' ' -> 'at '
  Rule  7: 's' + ' ' -> 's '
  Rule  8: 'd' + 'og' -> 'dog'
  Rule  9: 'i' + 's ' -> 'is '
  Rule 10: 'the ' + 'c' -> 'the c'  <- how 'the' grows
  Rule 11: 'the c' + 'at ' -> 'the cat '  <- how 'the' grows
  Rule 12: ' ' + 'the ' -> ' the '
  Rule 13: 'n' + 'd' -> 'nd'
  Rule 14: 's' + 'at ' -> 'sat '  <- how 'the' grows
  Rule 15: 'sat ' + 'o' -> 'sat o'

See? 'the ' is not recognized in one shot. It grows as 't'+'h'->'th', then merges with 'e ', step by step.


---

## Summary

Checklist:

1. BPE starts from characters or bytes because the starting point must be robust
2. Each step counts adjacent pairs and merges the most frequent one
3. Merge rules have an order; encoding new text must replay them in that order
4. `num_merges` controls vocab size and how "intact" tokens become
5. Decode is simple: lookup IDs and concatenate strings
6. BPE avoids most OOV by falling back to smaller pieces
7. Numbers are also token sequences; for decimals the model must learn digit alignment instead of comparing surface tokens

If you remember only one picture: **BPE grows tokens from characters, layer by layer, into common subwords.**

Next -> **Part 03**: token IDs are just labels. A neural network cannot directly understand labels. How do we turn IDs into vectors? We'll enter Embedding + Position Encoding.


---

## Exercises: BPE Tokenizer

These 3 exercises fall into two categories:

1. **Core idea**: BPE repeatedly counts adjacent pairs and merges the most frequent pair.
2. **Modern practice**: real tokenizers often start from bytes and encode by replaying ordered merge rules.

> **About using AI**: it is fine to ask for hints like "what should I merge next?", but avoid having an AI complete the code for you. The point is to watch tokens grow step by step with your own eyes.


### Exercise 1: core mechanism - count adjacent pairs

The first step of BPE is counting frequencies of adjacent token pairs.

**Hint**: a pair is `(tokens[i], tokens[i + 1])`.


In [ ]:
# Exercise 1: count adjacent pairs
from collections import defaultdict

tokens = list("banana")
pair_counts = defaultdict(int)

# TODO: replace the triple-quoted placeholder with your code
"""Count all adjacent pairs here."""

assert pair_counts[("a", "n")] == 2, dict(pair_counts)
assert pair_counts[("n", "a")] == 2, dict(pair_counts)
assert pair_counts[("b", "a")] == 1, dict(pair_counts)
print("OK Exercise 1 passed: you remember BPE starts by counting adjacent pairs")


### Exercise 2: core mechanism - merge the most frequent pair

After finding a frequent pair, we merge it into a new token.

**Hint**: when you hit the pair, you consume two tokens at once; otherwise you consume one token.


In [ ]:
# Exercise 2: merge a pair
def merge_pair(tokens, pair):
    """Merge adjacent tokens that match `pair` into a new token."""
    merged = []
    i = 0
    while i < len(tokens):
        # TODO: replace the triple-quoted placeholder with your code
        """Decide whether you hit the pair, then update `merged` and `i`."""
    return merged

result = merge_pair(list("banana"), ("a", "n"))

assert result == ["b", "an", "an", "a"], result
print("OK Exercise 2 passed: you remember the core BPE move is merging")


### Exercise 3: modern practice - start from bytes

Modern tokenizers often start from bytes so any Unicode text has an entry point.

**Hint**: in Python, `text.encode("utf-8")` shows you the bytes of a string.


In [ ]:
# Exercise 3: UTF-8 bytes
text = "€😊"

# TODO: replace the triple-quoted placeholder with your code
byte_values = """Encode `text` into a list of UTF-8 byte values."""

assert not isinstance(byte_values, str), "Please replace the placeholder inside the triple quotes."
assert byte_values == [228, 189, 160, 240, 159, 152, 138], byte_values
print("OK Exercise 3 passed: you understand why modern tokenizers often start from bytes")
